# StyleVAR Quick Inference
Load the latest checkpoint and visualize style transfer results.

In [ ]:
import os, sys, glob, torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

# Add project root to path
ROOT = os.path.dirname(os.path.abspath('__file__'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Load Model

In [ ]:
from models import VQVAE, StyleVAR, build_vae_stylevar

# Build model
vae, var_model = build_vae_stylevar(
    V=4096, Cvae=32, ch=160, share_quant_resi=4,
    device=device,
    patch_nums=(1, 2, 3, 4, 5, 6, 8, 10, 13, 16),
    depth=20, shared_aln=False, attn_l2_norm=True,
    flash_if_available=True, fused_if_available=True,
    init_adaln=0.5, init_adaln_gamma=1e-5, init_head=0.02, init_std=-1,
    style_enc_dim=512,
)

# Load VAE
vae_ckpt = os.path.join(ROOT, 'ckpt', 'vae_ch160v4096z32.pth')
vae.load_state_dict(torch.load(vae_ckpt, map_location='cpu'), strict=True)
vae.eval()
print(f'VAE loaded from {vae_ckpt}')

# Load StyleVAR — try best/last checkpoint, fallback to initial ckpt
CKPT_CANDIDATES = [
    os.path.join(ROOT, 'Output', 'ar-ckpt-best.pth'),
    os.path.join(ROOT, 'Output', 'ar-ckpt-last.pth'),
    os.path.join(ROOT, 'local_output', 'ar-ckpt-best.pth'),
    os.path.join(ROOT, 'local_output', 'ar-ckpt-last.pth'),
    os.path.join(ROOT, 'ckpt', 'style_var_d20_11_20_21.pth'),
]

var_ckpt = None
for c in CKPT_CANDIDATES:
    if os.path.exists(c):
        var_ckpt = c
        break

assert var_ckpt is not None, f'No checkpoint found in {CKPT_CANDIDATES}'

ckpt = torch.load(var_ckpt, map_location='cpu')
# Handle wrapped checkpoints
if 'trainer' in ckpt and 'var_wo_ddp' in ckpt['trainer']:
    state = ckpt['trainer']['var_wo_ddp']
    print(f'Loaded from trainer state (epoch={ckpt.get("epoch", "?")}, iter={ckpt.get("iter", "?")})')
elif 'model' in ckpt:
    state = ckpt['model']
else:
    state = ckpt

var_model.load_state_dict(state, strict=True)
var_model.eval()
print(f'StyleVAR loaded from {var_ckpt}')

## 2. Prepare Data

In [ ]:
import random

NUM_SAMPLES = 8  # how many pairs to visualize

# Transform: resize to 256x256, normalize to [-1, 1]
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3),
])

def load_img(path):
    """Load image and return (1, 3, 256, 256) tensor in [-1, 1]."""
    img = Image.open(path).convert('RGB')
    return transform(img).unsqueeze(0).to(device)

def tensor_to_pil(t):
    """Convert (3, H, W) tensor in [0, 1] to PIL Image."""
    return Image.fromarray((t.clamp(0, 1).permute(1, 2, 0).cpu().numpy() * 255).astype('uint8'))

# --- Load sample data ---
DATA_ROOT = os.path.join(ROOT, 'data', 'OmniStyle-150K')
# DATA_ROOT = os.path.join(ROOT, 'data', 'ImagePulse')

content_dir = os.path.join(DATA_ROOT, 'content')
style_dir = os.path.join(DATA_ROOT, 'style')
target_dir_candidates = [
    os.path.join(DATA_ROOT, 'target'),
    os.path.join(DATA_ROOT, 'OmniStyle-150K'),
    os.path.join(DATA_ROOT, 'OmniStyle-150k'),
]
target_dir = next((d for d in target_dir_candidates if os.path.isdir(d)), None)

if target_dir and os.path.isdir(content_dir):
    # OmniStyle layout — random sample
    all_target_files = [f for f in os.listdir(target_dir) if '&&' in f]
    sampled_files = random.sample(all_target_files, min(NUM_SAMPLES * 3, len(all_target_files)))
    samples = []
    for tf in sampled_files:
        c_name, s_raw = tf.split('&&')
        s_name = s_raw[:-4]
        c_path = os.path.join(content_dir, c_name)
        s_path = os.path.join(style_dir, s_name)
        if os.path.isfile(c_path) and os.path.isfile(s_path):
            samples.append((c_path, s_path))
        if len(samples) >= NUM_SAMPLES:
            break
    print(f'Randomly selected {len(samples)} OmniStyle pairs')
else:
    # ImagePulse layout — random sample
    ip_root = os.path.join(ROOT, 'data', 'ImagePulse')
    all_dirs = [d for d in os.listdir(ip_root)
                if os.path.isdir(os.path.join(ip_root, d)) and d.isdigit()]
    sampled_dirs = random.sample(all_dirs, min(NUM_SAMPLES, len(all_dirs)))
    samples = []
    for sd in sampled_dirs:
        c = os.path.join(ip_root, sd, 'content.png')
        s = os.path.join(ip_root, sd, 'style.png')
        if os.path.isfile(c) and os.path.isfile(s):
            samples.append((c, s))
    print(f'Randomly selected {len(samples)} ImagePulse pairs')

print(f'First sample: {samples[0]}')

## 3. Generate & Visualize

In [ ]:
N = min(len(samples), 8)
fig, axes = plt.subplots(N, 3, figsize=(12, 4 * N))
if N == 1:
    axes = axes[None, :]

axes[0, 0].set_title('Content', fontsize=14)
axes[0, 1].set_title('Style', fontsize=14)
axes[0, 2].set_title('Generated', fontsize=14)

with torch.no_grad():
    for i in range(N):
        c_path, s_path = samples[i]
        content = load_img(c_path)
        style = load_img(s_path)

        # Generate
        gen = var_model.autoregressive_infer(
            B=1, style_img=style, content_img=content,
            top_k=900, top_p=0.96, g_seed=42,
        )  # (1, 3, 256, 256) in [0, 1]

        # Display
        c_show = content[0].mul(0.5).add(0.5).clamp(0, 1)  # [-1,1] -> [0,1]
        s_show = style[0].mul(0.5).add(0.5).clamp(0, 1)
        g_show = gen[0].clamp(0, 1)

        for ax in axes[i]:
            ax.axis('off')
        axes[i, 0].imshow(c_show.permute(1, 2, 0).cpu().numpy())
        axes[i, 1].imshow(s_show.permute(1, 2, 0).cpu().numpy())
        axes[i, 2].imshow(g_show.permute(1, 2, 0).cpu().numpy())

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'infer_results.png'), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved to {ROOT}/infer_results.png')

## 4. Style Strength Slider

In [ ]:
# Pick one sample and vary style_strength
c_path, s_path = samples[0]
content = load_img(c_path)
style = load_img(s_path)

strengths = [-0.8, -0.4, 0.0, 0.4, 0.8]
fig, axes = plt.subplots(1, len(strengths) + 2, figsize=(4 * (len(strengths) + 2), 4))

# Show content & style
axes[0].imshow(content[0].mul(0.5).add(0.5).clamp(0,1).permute(1,2,0).cpu().numpy())
axes[0].set_title('Content')
axes[0].axis('off')
axes[1].imshow(style[0].mul(0.5).add(0.5).clamp(0,1).permute(1,2,0).cpu().numpy())
axes[1].set_title('Style')
axes[1].axis('off')

with torch.no_grad():
    for j, ss in enumerate(strengths):
        gen = var_model.autoregressive_infer(
            B=1, style_img=style, content_img=content,
            top_k=900, top_p=0.96, g_seed=42,
            style_strength=ss,
        )
        axes[j + 2].imshow(gen[0].clamp(0,1).permute(1,2,0).cpu().numpy())
        axes[j + 2].set_title(f'strength={ss}')
        axes[j + 2].axis('off')

plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'style_strength_sweep.png'), dpi=150, bbox_inches='tight')
plt.show()